# 05 — Advanced Modeling (Exp 2→5)
**Purpose:** Incremental experiments adding datasets, optimizing per-target models.

- **Exp 2:** + External APIs (SoilGrids, Weather, Elevation, OSM)
- **Exp 3:** + Spatial Context (HydroATLAS, RiverATLAS, SANLC, WorldPop)
- **Exp 4:** Multi-Target Exploration (Separate vs RegressorChain vs MultiOutput)
- **Exp 5:** + Sentinel-2 satellite features

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor, RegressorChain
from sklearn.metrics import r2_score
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
WORK_DIR = '/kaggle/working'

import sys
sys.path.insert(0, '/kaggle/input/ey-water-quality-data/src')
from spatial_cv import LeaveStationGroupOut, evaluate_spatial_cv

In [ ]:
train = pd.read_parquet(f'{WORK_DIR}/train_featured.parquet')
val = pd.read_parquet(f'{WORK_DIR}/val_featured.parquet')

# Config — adjust these
TARGET_COLS = []  # auto-detect
for col in train.columns:
    cl = col.lower()
    if any(k in cl for k in ['alkalinity', 'conductance', 'phosphorus']):
        TARGET_COLS.append(col)

STATION_COL = 'GEMS_Station_Number'
META_COLS = [STATION_COL, 'Latitude', 'Longitude', 'Sample_Date', 'River_Name']

cv = LeaveStationGroupOut(n_splits=10, station_col=STATION_COL, random_state=SEED)

# Load feature sets from notebook 03
import yaml
try:
    with open(f'{WORK_DIR}/feature_sets.yaml', 'r') as f:
        feature_sets = yaml.safe_load(f)
    print('Loaded feature sets from YAML')
except:
    feature_sets = None
    print('No feature sets YAML found — using all numeric features')

# All numeric features (excluding meta + targets)
all_features = [c for c in train.select_dtypes(include=[np.number]).columns
                if c not in TARGET_COLS + META_COLS]
print(f'Total features available: {len(all_features)}')
print(f'Targets: {TARGET_COLS}')

## Exp 2: External API Features
Add SoilGrids, Weather, Elevation, OSM → measure marginal R² gain.

In [ ]:
print('=' * 60)
print('EXP 2: + External API Features')
print('=' * 60)

# Features that include external API data
exp2_features = all_features  # Use all features now (includes external)
print(f'Exp 2 features: {len(exp2_features)}')

exp2_results = {}
for target in TARGET_COLS:
    target_short = target.split('_')[0][:12]
    print(f'\n--- {target_short} ---')
    
    valid_mask = train[target].notna()
    X = train.loc[valid_mask, META_COLS + exp2_features].fillna(0)
    y = train.loc[valid_mask, target]
    
    model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                              random_state=SEED, n_jobs=-1, verbosity=0)
    result = evaluate_spatial_cv(model, X, y, cv)
    exp2_results[target_short] = result['mean_score']

print(f'\nExp 2 Mean R² = {np.mean(list(exp2_results.values())):.4f}')

## Exp 3: + Spatial Context Features
Add HydroATLAS, RiverATLAS, SANLC, WorldPop (if extracted).

In [ ]:
# Same as Exp 2 but with spatial context features added
# If HydroATLAS etc. were extracted in notebook 01, they're already in train_featured
print('=' * 60)
print('EXP 3: + Spatial Context Features')
print('=' * 60)
print('(Same feature set as Exp 2 if spatial data is already merged)')
print('Compare with Exp 2 results to measure marginal gain of spatial context)')

## Hyperparameter Tuning with Optuna (Per-Target)

In [ ]:
def optuna_xgb_objective(trial, X_train, y_train, cv, meta_cols):
    """Optuna objective for XGBoost hyperparameter search."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': 0,
    }
    
    model = xgb.XGBRegressor(**params)
    feature_cols = [c for c in X_train.columns if c not in meta_cols]
    
    scores = []
    for train_idx, test_idx in cv.split(X_train):
        model_clone = xgb.XGBRegressor(**params)
        model_clone.fit(X_train.iloc[train_idx][feature_cols], y_train.iloc[train_idx])
        preds = model_clone.predict(X_train.iloc[test_idx][feature_cols])
        scores.append(r2_score(y_train.iloc[test_idx], preds))
    
    return np.mean(scores)

In [ ]:
# Run Optuna for each target
N_TRIALS = 100  # increase for better results (200-300)
best_models = {}

for target in TARGET_COLS:
    target_short = target.split('_')[0][:12]
    print(f'\n{"="*50}')
    print(f'Optuna Tuning: {target_short} ({N_TRIALS} trials)')
    print(f'{"="*50}')
    
    valid_mask = train[target].notna()
    X_opt = train.loc[valid_mask, META_COLS + all_features].fillna(0).reset_index(drop=True)
    y_opt = train.loc[valid_mask, target].reset_index(drop=True)
    
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
    study.optimize(
        lambda trial: optuna_xgb_objective(trial, X_opt, y_opt, cv, META_COLS),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )
    
    print(f'  Best R²: {study.best_value:.4f}')
    print(f'  Best params: {study.best_params}')
    
    best_models[target_short] = {
        'best_score': study.best_value,
        'best_params': study.best_params,
        'study': study
    }

## Exp 4: Multi-Target Exploration
Compare: Separate (A) vs MultiOutput (B) vs RegressorChain (C) vs Native (D)

In [ ]:
print('=' * 60)
print('EXP 4: MULTI-TARGET EXPLORATION')
print('=' * 60)

# Prepare multi-target data
valid_mask = train[TARGET_COLS].notna().all(axis=1)
X_mt = train.loc[valid_mask, META_COLS + all_features].fillna(0).reset_index(drop=True)
y_mt = train.loc[valid_mask, TARGET_COLS].reset_index(drop=True)
feature_cols = [c for c in all_features]

base_xgb = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                              random_state=SEED, n_jobs=-1, verbosity=0)

strategies = {
    'A_Separate': None,  # Already computed above
    'B_MultiOutput': MultiOutputRegressor(base_xgb),
    'C_Chain_fwd': RegressorChain(base_xgb, order=[0, 1, 2]),  # Alk→EC→DRP
    'C_Chain_rev': RegressorChain(base_xgb, order=[2, 1, 0]),  # DRP→EC→Alk
    'D_Native_ETR': ExtraTreesRegressor(n_estimators=200, max_depth=30, random_state=SEED, n_jobs=-1),
}

exp4_results = {}

for strat_name, model in strategies.items():
    if model is None:
        print(f'\n{strat_name}: See individual model results above')
        continue
    
    print(f'\n--- {strat_name} ---')
    
    fold_scores = {t: [] for t in TARGET_COLS}
    
    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X_mt)):
        from sklearn.base import clone
        m = clone(model)
        m.fit(X_mt.iloc[train_idx][feature_cols], y_mt.iloc[train_idx])
        preds = m.predict(X_mt.iloc[test_idx][feature_cols])
        
        if preds.ndim == 1:  # single output wrapped as multi
            preds = preds.reshape(-1, 1)
        
        for t_idx, target in enumerate(TARGET_COLS):
            score = r2_score(y_mt.iloc[test_idx][target], preds[:, t_idx])
            fold_scores[target].append(score)
    
    mean_scores = {t.split('_')[0][:12]: np.mean(s) for t, s in fold_scores.items()}
    overall_mean = np.mean(list(mean_scores.values()))
    
    exp4_results[strat_name] = mean_scores
    exp4_results[strat_name]['MEAN'] = overall_mean
    
    print(f'  Per-target R²: {mean_scores}')
    print(f'  Mean R²: {overall_mean:.4f}')

print(f'\n\n=== EXP 4 COMPARISON ===')
exp4_df = pd.DataFrame(exp4_results).T
display(exp4_df.sort_values('MEAN', ascending=False))

## Summary: Best Models Per Target

In [ ]:
print('\n=== DEVLOG ENTRY: Exp 2-5 ===')
print(f'Exp 2: External APIs added')
for t, s in exp2_results.items():
    print(f'  {t}: R² = {s:.4f}')
print(f'  Mean R² = {np.mean(list(exp2_results.values())):.4f}')

print(f'\nExp 4: Multi-target comparison')
if exp4_results:
    for strat, scores in exp4_results.items():
        print(f'  {strat}: Mean R² = {scores.get("MEAN", 0):.4f}')

print(f'\nBest per-target models (Optuna-tuned):')
for t, info in best_models.items():
    print(f'  {t}: R² = {info["best_score"]:.4f}')